# Feature Engineering: pronostico de rendimiento de maiz

Este notebook compara cutoffs de 25%, 50% y 75% de la temporada local y evalua features agronomicamente motivadas. La logica reside en `src/maize_forecasting/feature_engineering.py`; este notebook solo ejecuta y presenta resultados.

## Protocolo registrado

- Objetivo: rendimiento final (`yield`, t/ha) por `adm_id x campaña`.
- Cohorte: campañas con calendario entre 2003 y 2022, igual para los tres cutoffs.
- Validacion rolling-origin: cuatro folds expansivos; validan 2009-2018 en bloques y entrenan solo con campañas anteriores.
- Holdout exploratorio: 2019-2022. Ya se habia consultado en una ejecucion inicial, por lo que no se presenta como test final independiente.
- Cutoff: fraccion de la ventana estatica SOS/EOS local; solo se usan observaciones fechadas hasta el cutoff.
- Seleccion: regla de un error estandar para cutoff y tamaño del bundle, basada en macro-MAE OOF por campaña.
- Baselines: mediana global de entrenamiento y modelo de clima acumulado hasta el cutoff.

Las unidades provisionales o no documentadas (precipitacion, VPD, AWC y codigos de drenaje) se excluyen de los candidatos principales. El CWB, que pasa `cwb = prec - et0`, se usa como balance hidrico. No se filtran rendimientos cero ni extremos.

In [ ]:
import json
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "src").is_dir():
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(ROOT_DIR / "src"))

from maize_forecasting.feature_engineering import prepare_model_input_candidates
from maize_forecasting.star_schema import query_star

## Preparacion de la tabla de entrada

Se carga el reporte de Feature Engineering ya evaluado y se prepara `model_input_candidates_cutoff_50`, agregando AWC, coordenadas y drenaje como columnas candidatas pendientes de evaluacion. Esta ejecucion sincroniza metadata y prepara la tabla, pero no ajusta estimadores ni entrena un modelo.

In [ ]:
feature_report_path = ROOT_DIR / "reports" / "feature_engineering" / "feature_engineering_report.json"
feature_report = json.loads(feature_report_path.read_text(encoding="utf-8"))
evaluation = feature_report["evaluation"]
model_input_report = prepare_model_input_candidates(overwrite=True)
print("Evaluacion previa:", feature_report["status"])
print("Cutoff elegido:", evaluation["selected_cutoff_pct"], "%")
print("Features seleccionadas previamente:", evaluation["selected_features"])
print("Tabla preparada:", model_input_report["table"], "filas:", model_input_report["rows"])
print("Entrenamiento ejecutado:", model_input_report["model_training_performed"])

## Cohorte y cobertura

Primero se comprueba que los checkpoints usan exactamente las mismas campañas y se muestra la distribucion del target de entrenamiento, incluidos sus ceros.

In [ ]:
display(pd.DataFrame(feature_report["cutoff_candidate_tables"]))
display(pd.DataFrame([evaluation["target_cohort"]["target_distribution"]]))
display(pd.DataFrame([
    {"split": split, "campaign_rows": rows}
    for split, rows in evaluation["target_cohort"]["temporal_split_counts"].items()
]))

## Comparacion objetiva de cutoffs

Los cutoffs y bundles pre-registrados se comparan conjuntamente en cuatro folds rolling-origin. Se aplica la regla de un error estandar: entre resultados cercanos al mejor macro-MAE OOF, se favorece un cutoff mas temprano y el bundle con menos variables. El holdout 2019-2022 no se usa para elegir.

In [ ]:
cutoff_results = pd.DataFrame(evaluation["cutoff_validation_results"])
climate_results = pd.DataFrame(evaluation["climate_core_cutoff_validation_results"])
display(cutoff_results[["cutoff_pct", "best_bundle", "best_bundle_macro_season_mae", "climate_core_macro_season_mae", "all_candidates_macro_season_mae", "n_seasons"]])
display(climate_results[["cutoff_pct", "macro_season_mae", "mae", "rmse", "r2"]])
dummy_results = pd.DataFrame(evaluation["dummy_validation_results"])
display(dummy_results[["cutoff_pct", "macro_season_mae", "mae", "rmse", "r2"]])

## Hipotesis y valor incremental

Se compara el bundle climatico base con grupos recientes, humedad, vegetacion y contexto estatico sobre predicciones OOF rolling-origin. Luego se revisan ablaciones de una variable y correlaciones altas en validacion; son diagnosticos, no pruebas causales ni criterios basados en el holdout.

In [ ]:
bundle_results = pd.DataFrame(evaluation["feature_bundle_validation_results"])
display(bundle_results[["feature_bundle", "n", "macro_season_mae", "mae", "rmse", "r2"]])

ablation = pd.DataFrame(evaluation["feature_ablation_validation"])
display(ablation[["feature", "macro_season_mae_with_feature", "macro_season_mae_without_feature", "delta_mae_when_removed"]].sort_values("delta_mae_when_removed"))
display(pd.DataFrame(evaluation["highly_correlated_feature_pairs_validation"]))

## Holdout exploratorio

El bundle y cutoff se eligen con OOF rolling-origin de 2009-2018. El modelo se vuelve a ajustar con campañas de desarrollo hasta 2018 y se informa 2019-2022 como holdout exploratorio. Ese periodo ya se habia consultado para el bundle inicial completo, por lo que sus metricas no son un test virgen. El baseline es la mediana del target de entrenamiento.

In [ ]:
test_summary = pd.DataFrame([
    {"model": "selected_feature_bundle", **evaluation["exploratory_holdout_metrics_selected_candidate"]},
    {"model": "dummy_median", **evaluation["exploratory_holdout_metrics_dummy_baseline"]},
])
display(test_summary[["model", "n", "n_seasons", "macro_season_mae", "mae", "rmse", "r2"]])
print("Error regional exploratorio: mediana MAE =", round(evaluation["holdout_region_stability"]["median_region_mae"], 3), "; P90 =", round(evaluation["holdout_region_stability"]["p90_region_mae"], 3))
print("Predicciones negativas sin recortar:", evaluation["exploratory_holdout_negative_predictions"])

## Tabla candidata para el futuro entrenamiento

`model_input_candidates_cutoff_50` tiene una fila por region/campaña con target, cutoff y features seleccionadas. AWC, coordenadas y `drainage_class_code` se incluyen como columnas candidatas nuevas, aun sin evaluar; el codigo de drenaje se guarda como categoria, no como ordinal. Las decisiones y unidades estan en `model_input_column_metadata` y `reports/feature_engineering/model_input_preparation_report.json`. No se ha entrenado ni serializado un modelo.

In [ ]:
display(pd.DataFrame([
    {
        "feature": item["name"],
        "group": item["group"],
        "decision": item["evaluation_decision"],
        "validation_ablation_delta": (item["validation_ablation"] or {}).get("delta_mae_when_removed"),
    }
    for item in feature_report["feature_decisions"]
]))
model_preview = query_star(
    "SELECT adm_id, season_label, prediction_date, target_yield, cwb_sum_to_cutoff, latest_ndvi, latest_fpar, available_water_capacity_cm_m, drainage_class_code, latitude_deg, longitude_deg FROM model_input_candidates_cutoff_50 ORDER BY adm_id, season_key LIMIT 10"
)
display(model_preview)
print("Feature tables: candidate_features_cutoff_25/50/75")
print("Input table: model_input_candidates_cutoff_50")
print("Evaluation table: feature_evaluation_results")
print("Feature registry:", feature_report["feature_registry"])